In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np
from astropy.table import Table, join
from astropy.time import Time

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
%matplotlib widget

import pandas as pd

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

import os
import pandas as pd
from matplotlib.cm import get_cmap

In [ ]:
os.environ["no_proxy"] += ",.consdb"

In [ ]:
url="http://consdb-pq.consdb:8080/consdb"

In [ ]:
consdb=ConsDbClient(url)

In [ ]:
# Query both consDB tables
exposure = consdb.query("SELECT * FROM cdb_latiss.exposure WHERE science_program = 'spec-survey' AND day_obs > 20250717 AND img_type = 'science'")

In [ ]:
len(exposure)

In [ ]:
exposure.columns

In [ ]:
current_target_list = ["HD009051", "HD14943", "HD031128", "HD36780", "HD38949", "HD42525", "HD60753", "HD73495", "HD111980", "HD115169", "HD132096", "HD142331", "HD146233", "HD167060", "HD200654", "HD205905"]

In [ ]:
# ─── Convert to DataFrame ────────────────────────────────────────────────
df = exposure.to_pandas()

# ─── Exclude unwanted targets ────────────────────────────────────────────
exclude_targets = ["HD   2811", "HD   3883", "HD 187818", "HD 186819"]
df = df[~df["target_name"].isin(exclude_targets)]

print(len(df))

# ─── Per-night stats ─────────────────────────────────────────────────────
grouped = df.groupby(["target_name", "day_obs"])["airmass"]
airmass_stats = grouped.agg(["min", "max"])
airmass_stats["range"] = airmass_stats["max"] - airmass_stats["min"]
airmass_stats = airmass_stats.reset_index()

# ─── Target-level summary ────────────────────────────────────────────────
nights_exceeding = (
    airmass_stats[airmass_stats["range"] > 0.8]
    .groupby("target_name")["day_obs"]
    .nunique()
    .reset_index(name="nights_exceeding 0.8 airmass range")
)

total_nights = (
    airmass_stats.groupby("target_name")["day_obs"]
    .nunique()
    .reset_index(name="total_nights")
)

summary = pd.merge(total_nights, nights_exceeding, on="target_name", how="left").fillna(0)
summary["nights_exceeding 0.8 airmass range"] = summary["nights_exceeding 0.8 airmass range"].astype(int)

# ─── Show results ────────────────────────────────────────────────────────
print("\nTarget-level summary:")
print(summary)

# ─── Plot 1: Airmass range per night for each target ─────────────────────
plt.figure(figsize=(12, 6))
for target, group in airmass_stats.groupby("target_name"):
    plt.plot(group["day_obs"], group["range"], marker="o", linestyle="-", label=target)

plt.axhline(0.8, color="red", linestyle="--", label="Threshold = 0.8")
plt.xlabel("Night (day_obs)")
plt.ylabel("Airmass Range (max - min)")
plt.title("Per-night Airmass Ranges by Target (Excluded: HD 2811, HD 3883, HD 187818)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# ─── Plot 2: Bar chart of nights exceeding 0.8 per target ────────────────
plt.figure(figsize=(10, 5))
summary.set_index("target_name")["nights_exceeding 0.8 airmass range"].sort_values().plot(kind="barh")
plt.xlabel("Number of Nights Exceeding 0.8 Range")
plt.ylabel("Target")
plt.title("Nights with Airmass Range > 0.8 per Target")
plt.tight_layout()
plt.show()

In [ ]:
len(df['day_obs'].unique())

In [ ]:
df['day_obs'].unique()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from astropy.time import Time

def _coerce_time_to_datetime(series, guess_mjd=False):
    """Return a pandas datetime64[ns, UTC] Series from MJD or ISO/datetime input."""
    if np.issubdtype(series.dtype, np.number) or "mjd" in str(series.name).lower() or guess_mjd:
        try:
            t = Time(series.values, format="mjd")
            return pd.to_datetime(t.to_datetime())
        except Exception:
            pass
    return pd.to_datetime(series, errors="coerce", utc=True)

def _slugify(name: str) -> str:
    """Convert arbitrary string into safe filename."""
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in str(name)).strip("_")

def _hhmm_formatter_factory(start_hour):
    """Formatter for x-axis showing HH:MM labels given start_hour wrap."""
    shift = int(start_hour) * 3600
    def _fmt(x, pos):
        total_seconds = (x + shift) % (24 * 3600)
        h = int(total_seconds // 3600)
        m = int((total_seconds % 3600) // 60)
        return f"{h:02d}:{m:02d}"
    return FuncFormatter(_fmt)

def make_airmass_overplot_by_time(
    df: pd.DataFrame,
    *,
    target_col="target_name",
    night_col="day_obs",
    time_col="exp_midpt",
    airmass_col="airmass",
    save_dir="airmass_overplot_by_target",
    targets=None,
    linewidth=1.4,
    line_alpha=0.45,
    marker="o",
    markersize=4.0,
    markeredgewidth=1.2,
    y_limits=(1.0, 2.5),
    invert_y=True,
    dpi=150,
    start_hour=22,
    end_hour=11,
):
    """
    Plot airmass vs. UTC time-of-day for each target, overlaying all nights on one frame.
    The x-axis is wrapped (start_hour → end_hour), and the legend is always shown at
    the bottom of the figure with space reserved so it never overlaps axis labels.
    """
    df = df.copy()
    needed = {target_col, night_col, time_col, airmass_col}
    missing = needed - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns in DataFrame: {missing}")

    # Convert time column to datetime
    df["__dt__"] = _coerce_time_to_datetime(df[time_col], guess_mjd=("mjd" in time_col.lower()))
    df = df.dropna(subset=["__dt__", airmass_col])

    if targets is not None:
        df = df[df[target_col].isin(targets)]
    if df.empty:
        print("No rows to plot after filtering.")
        return

    # Convert to seconds since UTC midnight and wrap around start_hour
    dt = df["__dt__"].dt
    sec_utc = (dt.hour * 3600 + dt.minute * 60 + dt.second + (dt.microsecond / 1e6)).astype(float)
    shift = start_hour * 3600
    df["__sec_shifted__"] = (sec_utc - shift) % (24 * 3600)

    # Define x-axis limits
    if end_hour < start_hour:
        x_min, x_max = 0, (end_hour + 24 - start_hour) * 3600
    else:
        x_min, x_max = 0, (end_hour - start_hour) * 3600

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    # Loop through each target and plot
    for tgt, g in df.groupby(target_col, sort=True):
        if g.empty:
            continue
        g = g.sort_values("__sec_shifted__")

        nights = sorted(g[night_col].dropna().unique().tolist())
        if not nights:
            continue

        cmap = get_cmap("tab20") if len(nights) <= 20 else get_cmap("tab20b")
        colors = {n: cmap(i % cmap.N) for i, n in enumerate(nights)}

        fig = plt.figure(figsize=(12, 6), dpi=dpi)
        ax = plt.gca()

        for n in nights:
            gn = g[g[night_col] == n]
            if gn.empty:
                continue
            c = colors[n]
            ax.plot(
                gn["__sec_shifted__"], gn[airmass_col],
                linestyle="-",
                linewidth=linewidth,
                alpha=line_alpha,
                color=c,
                marker=marker,
                markersize=markersize,
                markerfacecolor="none",
                markeredgecolor=c,
                markeredgewidth=markeredgewidth,
                label=str(n),
                zorder=2,
            )

        # X-axis formatting
        ax.xaxis.set_major_formatter(_hhmm_formatter_factory(start_hour))
        ax.xaxis.set_major_locator(MultipleLocator(2 * 3600))  # every 2 hours
        ax.set_xlim(x_min, x_max)
        ax.tick_params(axis="x", labelrotation=45)

        # Y-axis formatting
        if y_limits is not None:
            ax.set_ylim(*y_limits)
        if invert_y:
            ax.invert_yaxis()

        ax.set_xlabel(f"UTC Time (HH:MM) from {start_hour:02d}:00 to {end_hour:02d}:00")
        ax.set_ylabel("Airmass")
        ax.set_title(f"Airmass vs UTC Time — All Nights Overlaid — {tgt}", size="x-large")
        ax.grid(True, linestyle=":", alpha=0.5)

        # Legend always at bottom
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(
                handles, labels,
                title="day_obs",
                loc="upper center",
                bbox_to_anchor=(0.5, -0.25),  # below the plot
                ncol=8 if len(labels) > 16 else max(1, len(labels) // 2),
                fontsize=8,
                title_fontsize=9,
                frameon=False,
            )
            fig.subplots_adjust(bottom=0.32)  # reserve space for legend

        fig.tight_layout()

        # Save and show
        if save_dir is not None:
            out = os.path.join(save_dir, f"{_slugify(tgt)}.png")
            fig.savefig(out, dpi=dpi, bbox_inches="tight")

        plt.show()


In [ ]:
make_airmass_overplot_by_time(df, time_col="exp_midpt")